# 06 — TTS: XTTS-v2 Fine-tuning on Vietnamese Dialects

Fine-tunes Coqui XTTS-v2 on ViMD audio for dialect-conditioned TTS.

**Target KPIs**: MOS ≥ 4.3 | Speaker Similarity ≥ 0.90


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/VietSpeech'
AUDIO_DIR = f'{PROJECT_ROOT}/processed/audio'
CHECKPOINT_DIR = f'{PROJECT_ROOT}/checkpoints/tts'
EXPORT_DIR = f'{PROJECT_ROOT}/weights/xtts-v2'

import sys, os, torch
sys.path.insert(0, PROJECT_ROOT)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(EXPORT_DIR, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

In [ ]:
!pip install -q TTS
!pip install -q mlflow
print('TTS library installed')

In [ ]:
# ── Verify GPU memory ─────────────────────────────────────────────────────
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {vram_gb:.1f} GB')
    if vram_gb < 14:
        print('⚠️  WARNING: XTTS-v2 fine-tuning requires ~14GB VRAM.')
        print('   Consider using Colab Pro with A100 or reducing batch_size to 2.')
    else:
        print('✓ VRAM sufficient for XTTS-v2 fine-tuning')

In [ ]:
# ── Prepare training metadata ─────────────────────────────────────────────
# XTTS-v2 fine-tuning expects a metadata CSV:
# audio_file|text|speaker_name

import pandas as pd
import glob

# Load ViMD processed splits
df = pd.read_parquet(f'{PROJECT_ROOT}/processed/features/train_features.parquet')

# Create XTTS metadata format
xtts_metadata = pd.DataFrame({
    'audio_file': df['audio_path'],
    'text': df['transcript'],
    'speaker_name': df['province'],  # Use province as speaker name
})

metadata_path = f'{CHECKPOINT_DIR}/metadata.csv'
xtts_metadata.to_csv(metadata_path, sep='|', index=False, header=False)
print(f'XTTS metadata saved: {len(xtts_metadata)} utterances → {metadata_path}')
xtts_metadata.head()

In [ ]:
# ── XTTS-v2 Fine-tuning ───────────────────────────────────────────────────
from TTS.bin.train_tts import main as train_tts
from TTS.config.shared_configs import BaseDatasetConfig
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import XttsArgs, XttsAudioConfig, Xtts
from TTS.utils.trainer import Trainer, TrainerArgs

# Dataset config
dataset_config = BaseDatasetConfig(
    formatter='ljspeech',
    meta_file_train='metadata.csv',
    path=CHECKPOINT_DIR,
    language='vi',
)

# Audio config
audio_config = XttsAudioConfig(
    sample_rate=22050,
    dvae_sample_rate=22050,
    output_sample_rate=24000,
)

# Model config
model_args = XttsArgs(
    use_dvae_training=True,
)

config = XttsConfig(
    output_path=CHECKPOINT_DIR,
    model_args=model_args,
    audio=audio_config,
    run_name='vietspeech-xtts',
    project_name='VietSpeech',
    run_description='XTTS-v2 fine-tuned on Vietnamese 63 dialects from ViMD',
    dashboard_logger='mlflow',
    logger_uri='mlflow',
    audio_max_length=33.075 * 22050,
    batch_size=2 if vram_gb < 16 else 4,
    batch_group_size=48,
    eval_batch_size=2,
    num_loader_workers=4,
    eval_split_max_size=256,
    print_step=50,
    plot_step=100,
    log_model_step=1000,
    save_step=1000,
    save_n_checkpoints=5,
    save_checkpoints=True,
    epochs=100,
    datasets=[dataset_config],
)

model = Xtts.init_from_config(config)

# Load pretrained XTTS-v2 base weights
print('Loading XTTS-v2 base weights...')
model.load_checkpoint(config, checkpoint_dir=None, eval=False)
model.cuda() if device == 'cuda' else model.cpu()

trainer = Trainer(
    TrainerArgs(
        restore_path=None,
        use_accelerate=False,
        mixed_precision='fp16',
    ),
    config,
    output_path=CHECKPOINT_DIR,
    model=model,
    train_samples=xtts_metadata.to_dict('records'),
)

trainer.fit()
print('\nTTS Training complete!')

In [ ]:
# ── Export fine-tuned model ───────────────────────────────────────────────
import shutil

# Copy best checkpoint to export dir
best_checkpoint = sorted(glob.glob(f'{CHECKPOINT_DIR}/best_model*.pth'))[-1]
shutil.copy(best_checkpoint, f'{EXPORT_DIR}/model.pth')
shutil.copy(f'{CHECKPOINT_DIR}/config.json', f'{EXPORT_DIR}/config.json')

# Extract speaker embeddings per province for dialect conditioning
embed_dir = f'{EXPORT_DIR}/speaker_embeddings'
os.makedirs(embed_dir, exist_ok=True)

from TTS.api import TTS as TTSApi
import numpy as np

tts_api = TTSApi(model_path=f'{EXPORT_DIR}/model.pth', config_path=f'{EXPORT_DIR}/config.json')

# Extract one speaker embedding per province
for province in df['province'].unique():
    sample_audio = df[df['province'] == province]['audio_path'].iloc[0]
    if os.path.exists(sample_audio):
        gpt_cond_latent, speaker_embedding = tts_api.synthesizer.tts_model.get_conditioning_latents(
            audio_path=[sample_audio]
        )
        np.save(f'{embed_dir}/{province}.npy', speaker_embedding.cpu().numpy())

print(f'Exported model + {len(os.listdir(embed_dir))} speaker embeddings to {EXPORT_DIR}')

## ✓ XTTS-v2 TTS Fine-tuning Complete

**Model saved to**: `{EXPORT_DIR}/`

**To use locally**: Copy `VietSpeech/weights/xtts-v2/` to `storage/weights/xtts-v2/`

**Next**: Run `07_voice_clone.ipynb`